<a href="https://colab.research.google.com/github/mlister3/UMR_Momentum/blob/main/MomentumData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Dependencies
import pandas as pd
from numpy import object_
import os
import requests
from google.colab import auth
from googleapiclient.discovery import build
import warnings
from io import BytesIO
# Disable all warnings
warnings.filterwarnings("ignore")

# Set Auth to False
g_drive_auth = False

In [ ]:
def auth_user_g_drive():
  print("Initiating Google Drive authentication...")
  try:
    auth.authenticate_user()
    print("Google Drive authenticated successfully.")
    global g_drive_auth
    g_drive_auth = True
  except Exception as e:
    print(f"Authentication failed: {e}")

def request_download_file(fileId):
  service = build('drive', 'v3')
  try:
    # Request the file content
    request = service.files().get_media(fileId=fileId)
    downloaded_file = request.execute()
    file_content = BytesIO(downloaded_file)
    print(f"Successfully downloaded file with ID: {fileId}")
    return file_content
  except Exception as e:
    print(f"Failed to download file with ID {fileId}: {e}")
    print("Please check the file ID and ensure your Google account has access to the file.")

# Authenticate User Google Drive
auth_user_g_drive()

past_cohort_apps_df = pd.read_csv(request_download_file("1fYGcgEhcWTzYfn-NFda4lNhbjQQ-WOlJ"))

In [ ]:
Working_df = past_cohort_apps_df.copy()
Working_df['UMNID'] = Working_df['UMNID'].astype(str).str.zfill(7)
Working_Hot_df = Working_df.copy()
Working_Hot_df = Working_Hot_df.drop(['UMNID'], axis=1)

# Convert specified columns to object (string) type
object_cols = ['HS ZIP', 'High School Code', 'Person Native Pacific Islander Origin',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_0',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_1',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_2',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_3',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_4',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_5',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_6',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_7',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_8',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_9',
               'CourseWork_25Cohort_Table.Courses_All_Table.cat_course_grade_level_10']
# drop proceeding '.0'
for col in object_cols:
    Working_Hot_df[col] = Working_Hot_df[col].astype(str).str.rstrip('.0')
    Working_Hot_df[col] = Working_Hot_df[col].astype(str)

# Convert specified columns to boolean type
bool_cols = ['FGEN SG', 'FGEN RC', 'Slate FGEN Person',
             'Slate FGEN App', 'Help_Y', 'placementResults_isProctored_0']
for col in bool_cols:
    Working_Hot_df[col] = Working_Hot_df[col].astype(bool)

# Convert specified columns to nullable integer type to handle NaN values
int_cols = ['Class Rank', 'Class Size', 'placementResults_placementIndex_0',
            'placementResults_placementIndex_1', 'placementResults_placementIndex_2',
            'placementResults_placementIndex_3', 'placementResults_placementIndex_4',
            'placementResults_nbPlacementsTaken_0', 'placementResults_nbPlacementsTaken_1',
            'progressInfo_lastLearningProgress_nbTopicsLearned_0',
            'placementResults_isTimedOut_0', 'placementResults_duration_0',
            'placementResults_duration_1', 'placementResults_duration_2',
            'placementResults_duration_3', 'placementResults_duration_4',
            'progressInfo_performance_nbTopicsMastered_0', 'progressInfo_performance_nbTopicsMastered_1',
            'progressInfo_performance_nbTopicsLearned_0', 'progressInfo_performance_nbTopicsLearned_1',
            'progressInfo_performance_totalNbTopics_0', 'studentInfo_totalTime_0',
            'studentInfo_totalTime_1', 'studentInfo_totalTime_2', 'studentInfo_totalTime_3', 'studentInfo_totalTime_4', 'studentInfo_totalTime_5',
            'knowledgeCheckInfo_duration_0', 'knowledgeCheckInfo_duration_1',
            'knowledgeCheckInfo_duration_2', 'knowledgeCheckInfo_duration_3', 'knowledgeCheckInfo_duration_4',
            'knowledgeCheckInfo_performance_nbTopicsMastered_0', 'knowledgeCheckInfo_performance_nbTopicsMastered_1',
            'knowledgeCheckInfo_performance_nbTopicsMastered_2', 'knowledgeCheckInfo_performance_nbTopicsMastered_3', 'knowledgeCheckInfo_performance_nbTopicsMastered_4',
            'knowledgeCheckInfo_performance_totalNbTopics_0', 'progressInfo_lastLearningProgress_duration_0',
            'progressInfo_lastLearningProgress_nbTopicsLearnedPerHour_0']

for col in int_cols:
    Working_Hot_df[col] = Working_Hot_df[col].astype('Int64') # Use nullable integer type

# Convert specificed columns to datetime type
datetime_cols = ['placementResults_startDate_0', 'placementResults_startDate_1',
                 'placementResults_startDate_2', 'placementResults_startDate_3', 'placementResults_startDate_4',
                 'placementResults_endDate_0', 'placementResults_endDate_1',
                 'placementResults_endDate_2', 'placementResults_endDate_3', 'placementResults_endDate_4',
                 'knowledgeCheckInfo_startDate_0', 'knowledgeCheckInfo_startDate_1',
                 'knowledgeCheckInfo_startDate_2', 'knowledgeCheckInfo_startDate_3',
                 'knowledgeCheckInfo_startDate_4', 'knowledgeCheckInfo_endDate_0', 'knowledgeCheckInfo_endDate_1',
                 'knowledgeCheckInfo_endDate_2', 'knowledgeCheckInfo_endDate_3',
                 'knowledgeCheckInfo_endDate_4']
for col in datetime_cols:
    Working_Hot_df[col] = pd.to_datetime(Working_Hot_df[col])
    # Extract year, month, and day components
    Working_Hot_df[col + '_year'] = Working_Hot_df[col].dt.year
    Working_Hot_df[col + '_month'] = Working_Hot_df[col].dt.month
    Working_Hot_df[col + '_day'] = Working_Hot_df[col].dt.day

# Drop the original datetime columns after extracting features
Working_Hot_df = Working_Hot_df.drop(columns=datetime_cols)

Working_Hot_df.head(3)

In [ ]:
for col, dtype in Working_Hot_df.dtypes.items():
    print(f"{col}: {dtype}")

In [ ]:
# Specify the row index you want to inspect
row_index = 210  # You can change this index to view different rows

# Check if the index is within the DataFrame's bounds
if row_index < len(Working_Hot_df):
    print(f"--- Values for row index {row_index} ---")
    for column in Working_Hot_df.columns:
        value = Working_Hot_df.loc[row_index, column]
        print(f"{column}: {value}")
else:
    print(f"Error: Row index {row_index} is out of bounds for the DataFrame.")

In [ ]:
dummy_application_df = pd.get_dummies(Working_Hot_df)
dummy_application_df.head(3)

In [ ]:
!pip install keras_tuner -q
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
# Split our preprocessed data into our features and target arrays
# Identify datetime columns to exclude from scaling
datetime_cols = [
    'placementResults_startDate_0', 'placementResults_startDate_1',
    'placementResults_startDate_2', 'placementResults_startDate_3', 'placementResults_startDate_4',
    'placementResults_endDate_0', 'placementResults_endDate_1',
    'placementResults_endDate_2', 'placementResults_endDate_3', 'placementResults_endDate_4',
    'knowledgeCheckInfo_startDate_0', 'knowledgeCheckInfo_startDate_1',
    'knowledgeCheckInfo_startDate_2', 'knowledgeCheckInfo_startDate_3',
    'knowledgeCheckInfo_startDate_4', 'knowledgeCheckInfo_endDate_0', 'knowledgeCheckInfo_endDate_1',
    'knowledgeCheckInfo_endDate_2', 'knowledgeCheckInfo_endDate_3',
    'knowledgeCheckInfo_endDate_4'
]

X = dummy_application_df.drop(['Help_Y'], axis=1, errors='ignore')
y = dummy_application_df['Help_Y'].astype(int) # Convert boolean target to integer

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

In [ ]:
from sklearn.impute import SimpleImputer

# Create an imputer that fills missing values with the mean of the column
# Fit only on the training data to avoid data leakage
imputer = SimpleImputer(strategy='mean')

# Create a StandardScaler instances
scaler = StandardScaler()

imputer.fit(X_train)
# Transform both training and testing data
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit the StandardScaler
X_scaler = scaler.fit(X_train_imputed)

# Scale the data
X_train_scaled = X_scaler.transform(X_train_imputed)
X_test_scaled = X_scaler.transform(X_test_imputed)

## Compile, Train, & Eval

In [ ]:
# Define the filepath for saving the weights
checkpoint_filepath = 'weights_checkpoint.weights.h5'

# Create the ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    save_freq='epoch'
)

In [ ]:
# Define the model
nn_model = tf.keras.models.Sequential()

# First hidden layer with a fixed number of units and activation
nn_model.add(tf.keras.layers.Dense(units=80, activation='relu', input_dim=X_train_scaled.shape[1]))

# Second hidden layer with a fixed number of units and activation
nn_model.add(tf.keras.layers.Dense(units=30, activation='relu'))

# Output layer
nn_model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Check the structure of the model
nn_model.summary()

# Compile the model
nn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the model
history = nn_model.fit(X_train_scaled, y_train, epochs=30, verbose=1, batch_size=16, callbacks=[checkpoint_callback])

In [ ]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn_model.evaluate(X_test_scaled,y_test,verbose=1, batch_size=16)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

# Export our model to keras file
nn_model.save('Init Attempt.keras')

## Hyperparameter Tuning with Keras Tuner

In [ ]:
# Function to create a new model for Keras Tuner
def build_model(hp):
    nn_model = tf.keras.models.Sequential()

    # Allow kerastuner to decide which activation function to use in hidden layers
    activation = hp.Choice('activation', ['relu', 'leaky_relu', 'elu', 'selu'])

    # Allow kerastuner to decide number of neurons in first layer
    nn_model.add(tf.keras.layers.Dense(units=hp.Int('first_units', min_value=1, max_value=90, step=5),
                                       activation=activation, input_dim=X_train_scaled.shape[1]))

    # Allow kerastuner to decide number of hidden layers and neurons in each layer
    for i in range(hp.Int('num_hidden_layers', 3, 9)):
        nn_model.add(tf.keras.layers.Dense(units=hp.Int('units_' + str(i), min_value=1, max_value=500, step=5),
                                           activation=activation))

    # Output layer: Allow tuner to choose between sigmoid and softmax
    output_activation = hp.Choice('output_activation', ['sigmoid', 'linear'])
    nn_model.add(tf.keras.layers.Dense(units=1, activation=output_activation))

    # Compile the model
    nn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    return nn_model

# Create a RandomSearch tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=100,
    overwrite=True)

# Perform hyperparameter search
tuner.search(X_train_scaled, y_train, epochs=15, validation_data=(X_test_scaled, y_test))

In [ ]:
# Get the best models from the tuner
best_hps = tuner.get_best_hyperparameters(num_trials=1)

# Build the best model
best_model = tuner.hypermodel.build(best_hps[0])

# Display the best hyperparameters
print("Best Hyperparameters found by Keras Tuner:")
for param, value in best_hps[0].values.items():
    print(f"  {param}: {value}")

# Summarize the best model
best_model.summary()

## Building and Training the Optimized Model

In [ ]:
# Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the optimized model using the best hyperparameters
nn_model_optimized = tuner.hypermodel.build(best_hps)

# Display the summary of the optimized model
nn_model_optimized.summary()

# Compile the optimized model (already done in build_model, but explicitly here for clarity)
nn_model_optimized.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the optimized model
history_optimized = nn_model_optimized.fit(X_train_scaled, y_train, epochs=30, verbose=2, batch_size=16, callbacks=[checkpoint_callback])

In [ ]:
# Evaluate the optimized model using the test data
model_loss_optimized, model_accuracy_optimized = nn_model_optimized.evaluate(X_test_scaled, y_test, verbose=1, batch_size=16)
print(f"Optimized Model Loss: {model_loss_optimized}, Optimized Model Accuracy: {model_accuracy_optimized}")

# Export our optimized model to an keras file
nn_model_optimized.save('Optimized_Model.keras')